# Document Categorization — Clustering Visualization

**Technical notebook** for the team/manager. Shows how the raw teaching documents
were understood and re-categorized by the Teaching Agent's functions, replacing the
inaccurate original folders.

Pipeline: extract each PDF → embed → cluster (KMeans) + score against the 6 function
categories → 2D visualization → editable mapping.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

## 1. Load the per-document JSON (from the pre-processing step)
Run first if needed: `python -m copilot.preprocess.doc_json ./data/teaching --out doc_json`

In [ ]:
import json, glob
docs = [json.load(open(p, encoding='utf-8')) for p in glob.glob('../doc_json/*.json') if not p.endswith('_index.json')]
print(f'{len(docs)} documents loaded')
docs[0]['source'] if docs else 'run doc_json first'

## 2. Embed each document

In [ ]:
from copilot.rag.embeddings import embed_texts
import numpy as np

texts = [d['source'] + '\n' + d.get('preview','') for d in docs]
vecs = np.array(embed_texts(texts, input_type='document'))
print('embedding matrix:', vecs.shape)

## 3. Cluster + reduce to 2D for visualization

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

k = 6
clusters = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(vecs)
coords = PCA(n_components=2, random_state=42).fit_transform(vecs)
print('reduced to', coords.shape)

## 4. Visualize the document clusters

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11,7))
scatter = plt.scatter(coords[:,0], coords[:,1], c=clusters, cmap='tab10', s=60, alpha=0.8)
for i, d in enumerate(docs):
    plt.annotate(d['source'][:22], (coords[i,0], coords[i,1]), fontsize=6, alpha=0.6)
plt.title('Teaching documents — semantic clusters', fontsize=13)
plt.xlabel('PCA 1'); plt.ylabel('PCA 2')
plt.colorbar(scatter, label='cluster')
plt.tight_layout(); plt.show()

## 5. Proposed categories (by teaching-agent function)
Run: `python -m copilot.preprocess.categorize --doc-json doc_json` to generate the CSV,
then load it here to see the distribution.

In [ ]:
import pandas as pd
df = pd.read_csv('../categorization_mapping.csv')
display(df['proposed_category'].value_counts())
df.head(15)

In [ ]:
# category distribution bar chart
df['proposed_category'].value_counts().plot(kind='barh', color='#EA5B0C', figsize=(9,4))
plt.title('Documents per teaching-agent function'); plt.xlabel('count')
plt.tight_layout(); plt.show()